In [1]:
import datasets
from datasets import ClassLabel
import numpy as np
import torch
from transformers import AutoTokenizer
from tqdm import tqdm
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd
from sklearn.metrics import balanced_accuracy_score, accuracy_score, f1_score
import plotly.io as pio

from kaleido.scopes.plotly import PlotlyScope

scope = PlotlyScope()
pio.kaleido.scope.chromium_args = tuple(
    [arg for arg in scope.chromium_args if arg != "--disable-dev-shm-usage"])

In [58]:
# load results

ds_common_vanilla = datasets.load_from_disk(
    "results/common_stories_16_roc_6_wp_percent_Vanilla_1_Cycles_0.0_WarmR_1e-05_lr_10_Epochs_in_faketruebr_scores"
)['validation']

ds_common_rst = datasets.load_from_disk(
    "results/common_stories_16_roc_6_wp_percent_RSTMix_1_Cycles_0.0_WarmR_1e-05_lr_10_Epochs_in_faketruebr_scores"
)['validation']

ds_common_pos = datasets.load_from_disk(
    "results/common_stories_16_roc_6_wp_percent_POSMix_1_Cycles_0.0_WarmR_1e-05_lr_10_Epochs_in_faketruebr_scores"
)['validation']

ds_common_br_vanilla = datasets.load_from_disk(
    "results/common_stories_16_roc_6_wp_percent_br_Vanilla_1_Cycles_0.0_WarmR_1e-05_lr_10_in_faketruebr_scores"
)['validation']

ds_common_br_rst = datasets.load_from_disk(
    "results/common_stories_16_roc_6_wp_percent_br_RSTMix_1_Cycles_0.0_WarmR_1e-05_lr_10_in_faketruebr_scores"
)['validation']

ds_common_br_pos = datasets.load_from_disk(
    "results/common_stories_16_roc_6_wp_percent_br_POSMix_1_Cycles_0.0_WarmR_1e-05_lr_10_in_faketruebr_scores"
)['validation']

ds_gcdc_vanilla = datasets.load_from_disk(
    "results/gcdc_Vanilla_1_Cycles_0.0_WarmR_in_faketruebr_scores"
)['validation']

ds_gcdc_rst = datasets.load_from_disk(
    "results/gcdc_RSTMix_1_Cycles_0.0_WarmR__in_faketruebr_scores"
)['validation']

ds_gcdc_pos = datasets.load_from_disk(
    "results/gcdc_POSMix_1_Cycles_0.0_WarmR__in_faketruebr_scores"
)['validation']

In [7]:
common_column = [
    'text_id', 'text', 'link', 'label', 'related_to', 'text_edus',
    'text_edus_breaks', 'text_rst', 'text_rst_mixed', 'text_pos_mixed'
]

In [18]:
set(ds_gcdc_pos.column_names) - set(common_column)

{'gcdc_POSMix_1_Cycles_0.0_WarmR_1e-05_lr_10_Epochs_run-0',
 'gcdc_POSMix_1_Cycles_0.0_WarmR_1e-05_lr_10_Epochs_run-1',
 'gcdc_POSMix_1_Cycles_0.0_WarmR_1e-05_lr_10_Epochs_run-2',
 'gcdc_POSMix_1_Cycles_0.0_WarmR_1e-05_lr_10_Epochs_run-3',
 'gcdc_POSMix_1_Cycles_0.0_WarmR_1e-05_lr_10_Epochs_run-4'}

In [47]:
def preds_with_probs(ds, columns):
  p = []
  for k in columns:
    preds = {
        'correct_ids': [],
        'correct_logits': [],
        'correct_lenght': [],
        'wrong_ids': [],
        'wrong_logits': [],
        'wrong_lenght': [],
        'label': [],
        'prediction': [],
    }
    for row_idx in tqdm(range(ds.num_rows)):
      predicted = np.argmax(ds[row_idx][k][0])
      label = ds[row_idx]['label']
      preds['prediction'].append(predicted)
      preds['label'].append(label)
      if predicted == label:
        preds['correct_ids'].append(row_idx)
        preds['correct_logits'].append(ds[row_idx][k][0][predicted])
        preds['correct_lenght'].append(int(ds[row_idx][k][1][0]))
      else:
        preds['wrong_ids'].append(row_idx)
        preds['wrong_logits'].append(ds[row_idx][k][0][predicted])
        preds['wrong_lenght'].append(int(ds[row_idx][k][1][0]))
    p.append(preds)
  return p

In [60]:
c_ds_common_vanilla = list(
    set(ds_common_vanilla.column_names) - set(common_column))
preds_ds_common_vanilla = preds_with_probs(ds_common_vanilla,
                                           c_ds_common_vanilla)
ba = []
for p_run in preds_ds_common_vanilla:
  ba.append(balanced_accuracy_score(p_run['label'], p_run['prediction']))
# print mean, median and std
print(np.mean(ba), np.median(ba), np.std(ba))

100%|██████████| 533/533 [00:01<00:00, 518.14it/s]


0.638154374700797 0.6478626341133733 0.020827321477269855


In [61]:
c_ds_common_rst = list(set(ds_common_rst.column_names) - set(common_column))
preds_ds_common_rst = preds_with_probs(ds_common_rst, c_ds_common_rst)
ba = []
for p_run in preds_ds_common_rst:
  ba.append(balanced_accuracy_score(p_run['label'], p_run['prediction']))
# print mean, median and std
print(np.mean(ba), np.median(ba), np.std(ba))

100%|██████████| 533/533 [00:00<00:00, 666.58it/s]

0.6964074512123004 0.6964074512123004 0.04114570133198159


In [62]:
c_ds_common_pos = list(set(ds_common_pos.column_names) - set(common_column))
preds_ds_common_pos = preds_with_probs(ds_common_pos, c_ds_common_pos)
ba = []
for p_run in preds_ds_common_pos:
  ba.append(balanced_accuracy_score(p_run['label'], p_run['prediction']))
# print mean, median and std
print(np.mean(ba), np.median(ba), np.std(ba))

100%|██████████| 533/533 [00:00<00:00, 635.45it/s]

0.6849764861592182 0.6629002280983357 0.0496341044402149


In [63]:
c_ds_common_br_vanilla = list(
    set(ds_common_br_vanilla.column_names) - set(common_column))
preds_ds_common_br_vanilla = preds_with_probs(ds_common_br_vanilla,
                                              c_ds_common_br_vanilla)
ba = []
for p_run in preds_ds_common_br_vanilla:
  ba.append(balanced_accuracy_score(p_run['label'], p_run['prediction']))
# print mean, median and std
print(np.mean(ba), np.median(ba), np.std(ba))

100%|██████████| 533/533 [00:00<00:00, 766.22it/s]

0.6317943735743854 0.6235603052575258 0.024451045285249884


In [64]:
c_ds_common_br_rst = list(
    set(ds_common_br_rst.column_names) - set(common_column))
preds_ds_common_br_rst = preds_with_probs(ds_common_br_rst, c_ds_common_br_rst)
ba = []
for p_run in preds_ds_common_br_rst:
  ba.append(balanced_accuracy_score(p_run['label'], p_run['prediction']))
# print mean, median and std
print(np.mean(ba), np.median(ba), np.std(ba))

100%|██████████| 533/533 [00:00<00:00, 1211.23it/s]

0.6807960913519755 0.6759173213933711 0.03608320007908102


In [69]:
c_ds_common_br_pos = list(
    set(ds_common_br_pos.column_names) - set(common_column))
preds_ds_common_br_pos = preds_with_probs(ds_common_br_pos, c_ds_common_br_pos)
ba = []
for p_run in preds_ds_common_br_pos:
  if balanced_accuracy_score(p_run['label'], p_run['prediction']) != 0.5:
    ba.append(balanced_accuracy_score(p_run['label'], p_run['prediction']))
# print mean, median and std
print(np.mean(ba), np.median(ba), np.std(ba))

100%|██████████| 533/533 [00:01<00:00, 523.51it/s]


0.759773380079412 0.7574378361634424 0.02131537379978216


In [70]:
c_ds_gcdc_vanilla = list(set(ds_gcdc_vanilla.column_names) - set(common_column))
preds_ds_gcdc_vanilla = preds_with_probs(ds_gcdc_vanilla, c_ds_gcdc_vanilla)
ba = []
for p_run in preds_ds_gcdc_vanilla:
  if balanced_accuracy_score(p_run['label'], p_run['prediction']) != 0.5:
    ba.append(balanced_accuracy_score(p_run['label'], p_run['prediction']))
# print mean, median and std
print(np.mean(ba), np.median(ba), np.std(ba))

100%|██████████| 533/533 [00:01<00:00, 466.30it/s]


0.6902529196184684 0.7313156486722424 0.08136145857719487


In [71]:
c_ds_gcdc_rst = list(set(ds_gcdc_rst.column_names) - set(common_column))
preds_ds_gcdc_rst = preds_with_probs(ds_gcdc_rst, c_ds_gcdc_rst)
ba = []
for p_run in preds_ds_gcdc_rst:
  if balanced_accuracy_score(p_run['label'], p_run['prediction']) != 0.5:
    ba.append(balanced_accuracy_score(p_run['label'], p_run['prediction']))
# print mean, median and std
print(np.mean(ba), np.median(ba), np.std(ba))

100%|██████████| 533/533 [00:00<00:00, 973.12it/s]

0.6118843456956999 0.6151614992537524 0.053578358794045454


In [68]:
c_ds_gcdc_pos = list(set(ds_gcdc_pos.column_names) - set(common_column))
preds_ds_gcdc_pos = preds_with_probs(ds_gcdc_pos, c_ds_gcdc_pos)
ba = []
for p_run in preds_ds_gcdc_pos:
  ba.append(balanced_accuracy_score(p_run['label'], p_run['prediction']))
# print mean, median and std
print(np.mean(ba), np.median(ba), np.std(ba))

100%|██████████| 533/533 [00:00<00:00, 1203.97it/s]

0.5887696770014925 0.5954915378333474 0.06032641584672246
